In [19]:
import polars as pl

In [20]:
demographic_data = pl.read_csv("cc-est2024-alldata.csv", encoding = "latin-1", schema_overrides = {
    "STATE": pl.String,
    "COUNTY": pl.String,
    "SUMLEV": pl.String,
    "AGEGRP": pl.String
})

demographic_data = demographic_data.with_columns(
    (pl.col("STATE") + pl.col("COUNTY")).alias("FIPS_CODE")
)

print(f"Total Rows: {demographic_data.select(pl.len()).item():,}")

Total Rows: 358,416


In [21]:
agegrp_map = {
    0: "Total",
    1: "Age 0 to 4 years",
    2: "Age 5 to 9 years",
    3: "Age 10 to 14 years",
    4: "Age 15 to 19 years",
    5: "Age 20 to 24 years",
    6: "Age 25 to 29 years",
    7: "Age 30 to 34 years",
    8: "Age 35 to 39 years",
    9: "Age 40 to 44 years",
    10: "Age 45 to 49 years",
    11: "Age 50 to 54 years",
    12: "Age 55 to 59 years",
    13: "Age 60 to 64 years",
    14: "Age 65 to 69 years",
    15: "Age 70 to 74 years",
    16: "Age 75 to 79 years",
    17: "Age 80 to 84 years",
    18: "Age 85 years or older"
}

# Map AGEGRP to labels
demographic_data = demographic_data.with_columns(
    pl.col("AGEGRP").replace(agegrp_map).alias("AGEGRP")
)

In [27]:
demographic_data_agg = (
    demographic_data
    .group_by(["FIPS_CODE", "AGEGRP"])
)

float_cols = [col for col, dtype in zip(demographic_data.columns, demographic_data.dtypes) if dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64] and col != "YEAR"]


demographic_data_agg = demographic_data_agg.agg(
    [pl.col(col).first() for col, dtype in zip(demographic_data.columns, demographic_data.dtypes) if dtype == pl.Utf8 and col not in ["FIPS_CODE", "AGEGRP"]] + 
    [pl.col(col).mean() for col in float_cols]

)

demographic_data_agg.head(1)

FIPS_CODE,AGEGRP,SUMLEV,STATE,COUNTY,STNAME,CTYNAME,TOT_POP,TOT_MALE,TOT_FEMALE,WA_MALE,WA_FEMALE,BA_MALE,BA_FEMALE,IA_MALE,IA_FEMALE,AA_MALE,AA_FEMALE,NA_MALE,NA_FEMALE,TOM_MALE,TOM_FEMALE,WAC_MALE,WAC_FEMALE,BAC_MALE,BAC_FEMALE,IAC_MALE,IAC_FEMALE,AAC_MALE,AAC_FEMALE,NAC_MALE,NAC_FEMALE,NH_MALE,NH_FEMALE,NHWA_MALE,NHWA_FEMALE,NHBA_MALE,…,NHNA_FEMALE,NHTOM_MALE,NHTOM_FEMALE,NHWAC_MALE,NHWAC_FEMALE,NHBAC_MALE,NHBAC_FEMALE,NHIAC_MALE,NHIAC_FEMALE,NHAAC_MALE,NHAAC_FEMALE,NHNAC_MALE,NHNAC_FEMALE,H_MALE,H_FEMALE,HWA_MALE,HWA_FEMALE,HBA_MALE,HBA_FEMALE,HIA_MALE,HIA_FEMALE,HAA_MALE,HAA_FEMALE,HNA_MALE,HNA_FEMALE,HTOM_MALE,HTOM_FEMALE,HWAC_MALE,HWAC_FEMALE,HBAC_MALE,HBAC_FEMALE,HIAC_MALE,HIAC_FEMALE,HAAC_MALE,HAAC_FEMALE,HNAC_MALE,HNAC_FEMALE
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""05097""","""Age 55 to 59 years""","""050""","""05""","""097""","""Arkansas""","""Montgomery County""",722.5,341.833333,380.666667,327.333333,341.333333,1.666667,2.166667,6.333333,4.666667,0.833333,27.166667,0.0,0.0,5.666667,5.333333,332.333333,346.166667,3.0,3.833333,10.5,8.666667,1.666667,27.333333,0.166667,0.166667,327.5,368.5,316.5,331.833333,0.333333,…,0.0,4.5,4.833333,320.5,336.333333,1.666667,2.0,8.5,7.833333,1.5,27.333333,0.0,0.0,14.333333,12.166667,10.833333,9.5,1.333333,1.666667,1.0,0.5,0.0,0.0,0.0,0.0,1.166667,0.5,11.833333,9.833333,1.333333,1.833333,2.0,0.833333,0.166667,0.0,0.166667,0.166667


In [29]:
cols_not_include = [
    "SUMLEV",
    "STATE",
    "COUNTY",
    "STNAME",
    "CTYNAME"
]

eavs_data = pl.read_csv("combined_county.csv", schema_overrides = {"fips_code": pl.String})

eavs_data.filter(pl.col("jurisdiction_name") == "RUSK COUNTY")

fips_code,jurisdiction_name,total_registrations_received,new_valid_registrations,rejected_registrations,total_forms_mail_fax_email,new_registrations_mail_fax_email,rejected_registrations_mail_fax_email,total_forms_in_person,new_registrations_in_person,rejected_registrations_in_person,total_forms_online,new_registrations_online,rejected_registrations_online,total_forms_dmv,new_registrations_dmv,rejected_registrations_dmv,total_forms_mandatory_nvra,new_registrations_mandatory_nvra,rejected_registrations_mandatory_nvra,total_forms_disability_agency,new_registrations_disability_agency,rejected_registrations_disability_agency,total_forms_armed_forces,new_registrations_armed_forces,rejected_registrations_armed_forces,total_forms_discretionary_nvra,new_registrations_discretionary_nvra,rejected_registrations_discretionary_nvra,total_forms_advocacy_groups,new_registrations_advocacy_groups,rejected_registrations_advocacy_groups,year,percent_rejected_registrations_total,percent_rejected_registrations_mail_fax_email,percent_rejected_registrations_in_person,percent_rejected_registrations_online,percent_rejected_registrations_dmv,percent_rejected_registrations_mandatory_nvra,percent_rejected_registrations_disability_agency,percent_rejected_registrations_armed_forces,percent_rejected_registrations_discretionary_nvra,percent_rejected_registrations_advocacy_groups,rejected_state_rank
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""4840100000""","""RUSK COUNTY""",8934.5,5292.0,27.5,2768.0,1559.5,10.0,145.0,129.5,0.5,45.0,14.0,0.0,5323.5,3158.5,10.5,397.0,234.0,5.0,4.5,3.0,0.0,12.0,6.5,0.0,156.0,104.0,1.5,0.0,0.0,0.0,2021.0,0.307796,0.641231,0.3861,0.0,0.332436,2.136752,0.0,0.0,1.442308,0.0,2171
